# 05 — Probability of Default (PD) Modeli: LightGBM

**Proje:** Credit Risk Scoring (Home Credit Default Risk)

Bu defterde mimarinin **2. katmanı** çalışıyor: müşterinin başvuru anındaki ~740 feature → **temerrüt olasılığı** (`predict_proba`).

### Akış
1. `train.parquet` ve `valid.parquet` yükle, `prepare_features` ile X/y'ye böl.
2. **LightGBM** binary classifier; AUC metric, **early stopping** valid set üzerinde.
3. **Değerlendirme:** AUC, Gini, KS, Log-Loss, Brier — kredi skorlama literatürünün temel metrikleri.
4. **ROC eğrisi** + **Calibration curve** (06 scorecard adımı için kalibrasyonun ne kadar iyi olduğunu görmek kritik).
5. **Feature importance** (gain) — top-30 görseli.
6. Model + valid predictions diske kaydedilir → `06_Scorecard.ipynb` bunları kullanır.

### Tasarım kararları
- `class_weight` veya `scale_pos_weight` **yok**; PD'nin ham olasılık yorumu bozulmasın diye. Skorlama adımında kalibrasyona bakacağız.
- Random seed = 42 (reproducible).
- Hiper-parametreler `src/modeling.py::DEFAULT_LGBM_PARAMS`'tan geliyor; bu defterde override etmiyoruz.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import PROCESSED_DIR
from src.modeling import (
    prepare_features,
    train_lightgbm,
    evaluate_pd_model,
    plot_roc,
    plot_calibration,
    plot_feature_importance,
    get_feature_importance,
    save_booster,
    DEFAULT_LGBM_PARAMS,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")

print(f"Processed dir : {PROCESSED_DIR}")

## 1) Veri yükleme

In [ ]:
train_df = pd.read_parquet(PROCESSED_DIR / "train.parquet")
valid_df = pd.read_parquet(PROCESSED_DIR / "valid.parquet")
print(f"train_df : {train_df.shape}")
print(f"valid_df : {valid_df.shape}")

X_train, y_train, feature_cols = prepare_features(train_df)
X_valid, y_valid, _ = prepare_features(valid_df)

cat_cols = [c for c in X_train.columns if isinstance(X_train[c].dtype, pd.CategoricalDtype)]
print(f"feature count    : {len(feature_cols)}")
print(f"categorical cols : {len(cat_cols)}")
print(f"\nTARGET ratio (train) : {y_train.mean():.4f}")
print(f"TARGET ratio (valid) : {y_valid.mean():.4f}")

## 2) LightGBM eğitimi

Hiper-parametre seti `DEFAULT_LGBM_PARAMS`'tan; n_estimators=2000, early_stopping_rounds=200 ile valid AUC durduğunda kesilir. Genelde 500-1000 iter aralığında durur.

In [ ]:
print("LightGBM params:")
for k, v in DEFAULT_LGBM_PARAMS.items():
    print(f"  {k:>20} = {v}")
print()

booster = train_lightgbm(
    X_train, y_train,
    X_valid, y_valid,
    num_boost_round=2000,
    early_stopping_rounds=200,
    log_period=100,
)

print(f"\nbest iteration : {booster.best_iteration}")
print(f"best score     : {booster.best_score['valid']['auc']:.6f}")

## 3) Değerlendirme metrikleri

- **AUC** : sıralama gücü (en kritik PD metriği)
- **Gini** = 2·AUC − 1 : aynı bilginin kredi sektörü standardı sunumu
- **KS**   : pozitif ve negatif sınıf CDF'leri arasındaki maksimum fark; kesim eşiği seçiminde kullanılır
- **Log-Loss** ve **Brier**: olasılık tahminlerinin **kalibrasyonu** hakkında bilgi verir

In [ ]:
y_pred_train = booster.predict(X_train, num_iteration=booster.best_iteration)
y_pred_valid = booster.predict(X_valid, num_iteration=booster.best_iteration)

metrics = pd.DataFrame(
    {
        "train": evaluate_pd_model(y_train, y_pred_train),
        "valid": evaluate_pd_model(y_valid, y_pred_valid),
    }
).round(4)
metrics

## 4) ROC ve Calibration eğrileri

Yan yana iki grafik:
- **ROC**: TPR vs FPR (üstüne diagonal "şans" çizgisi)
- **Calibration**: ham `predict_proba` ne kadar gerçeğin reel oranı? Diagonal'a yakınlık iyi kalibrasyondur.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_roc(y_valid, y_pred_valid, ax=axes[0], label="LightGBM (valid)")
plot_calibration(y_valid, y_pred_valid, n_bins=10, ax=axes[1])
plt.tight_layout()
plt.show()

## 5) Feature importance (gain)

Modelin hangi sinyallere ne kadar yaslandığını gösterir. **Beklenti**: `EXT_SOURCE_*`, `INSTAL_DPD_*`, `BURO_*`, `PREV_*` üst sıralarda.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 11))
plot_feature_importance(booster, top=30, importance_type="gain", ax=ax)
plt.tight_layout()
plt.show()

importance_df = get_feature_importance(booster, importance_type="gain")
print(f"\nTotal features used (importance > 0): {(importance_df['importance'] > 0).sum()}")
print(f"Top-50 share of total gain         : "
      f"{importance_df['importance'].head(50).sum() / importance_df['importance'].sum():.2%}")

## 6) Feature selection deneyleri

Baseline modelde 738 feature var, fakat **126 tanesi hiç kullanılmıyor** (importance = 0) ve **top-50 toplam gain'in ~%49**'unu, **top-100 ise ~%64**'ünü taşıyor. Train-valid AUC gap (~0.17) makul ama yüksek; daha az feature genelde **daha az variance + daha iyi generalization** verir.

### İki senaryo
- **A — `used`**: importance > 0 olan tüm feature'lar (~612). Ölü ağırlığı atar; teorik olarak metrikler değişmemeli, baseline doğrulaması.
- **B — `top200`**: gain'e göre en güçlü 200 feature. Daha agresif; hedef gap'i daraltmak.

Aynı `DEFAULT_LGBM_PARAMS` ve aynı seed ile yeniden eğitip karşılaştırıyoruz.

In [ ]:
import importlib
import src.modeling as _modeling
importlib.reload(_modeling)
from src.modeling import select_features_by_importance, train_lightgbm, evaluate_pd_model

features_used = select_features_by_importance(importance_df, min_importance=0.0)
features_top200 = select_features_by_importance(importance_df, top_n=200)

print(f"Scenario A (used)   : {len(features_used)} features")
print(f"Scenario B (top200) : {len(features_top200)} features")

scenarios = {
    "full":   list(X_train.columns),
    "used":   features_used,
    "top200": features_top200,
}

results = {
    "full": {
        "n_features":  len(X_train.columns),
        "best_iter":   booster.best_iteration,
        **{f"train_{k}": v for k, v in evaluate_pd_model(y_train, y_pred_train).items()},
        **{f"valid_{k}": v for k, v in evaluate_pd_model(y_valid, y_pred_valid).items()},
    }
}

trained_boosters = {"full": booster}

for name in ("used", "top200"):
    cols = scenarios[name]
    print(f"\n=== Training scenario: {name} ({len(cols)} features) ===")
    b = train_lightgbm(
        X_train[cols], y_train,
        X_valid[cols], y_valid,
        num_boost_round=2000,
        early_stopping_rounds=200,
        log_period=200,
    )
    trained_boosters[name] = b

    pred_tr = b.predict(X_train[cols], num_iteration=b.best_iteration)
    pred_va = b.predict(X_valid[cols], num_iteration=b.best_iteration)
    results[name] = {
        "n_features": len(cols),
        "best_iter":  b.best_iteration,
        **{f"train_{k}": v for k, v in evaluate_pd_model(y_train, pred_tr).items()},
        **{f"valid_{k}": v for k, v in evaluate_pd_model(y_valid, pred_va).items()},
    }

comparison = pd.DataFrame(results).T
comparison["auc_gap"] = (comparison["train_auc"] - comparison["valid_auc"]).round(4)
display_cols = ["n_features", "best_iter", "train_auc", "valid_auc", "auc_gap",
                "valid_gini", "valid_ks", "valid_log_loss", "valid_brier"]
comparison[display_cols].round(4)

## 7) Artifact'leri kaydet

**Kararlaştırılan model: `used` (612 feature)** — `full`'a göre AUC eşit, gap %23 düşük.

Sonraki defter (`06_Scorecard.ipynb`) için:
- `data/processed/lgbm_pd_model.txt` — LightGBM native format (used booster)
- `data/processed/valid_predictions.parquet` — `SK_ID_CURR`, `TARGET`, `pd_proba`
- `data/processed/lgbm_feature_importance.parquet` — final modelin gain importance'ı
- `data/processed/lgbm_selected_features.json` — used booster'ın eğitildiği feature isimleri (06'nın aynı sırayı kullanması için)

In [ ]:
import json

final_booster = trained_boosters["used"]
final_features = features_used

X_valid_final = X_valid[final_features]
y_pred_valid_final = final_booster.predict(
    X_valid_final, num_iteration=final_booster.best_iteration
)
final_importance_df = get_feature_importance(final_booster, importance_type="gain")

model_path = PROCESSED_DIR / "lgbm_pd_model.txt"
preds_path = PROCESSED_DIR / "valid_predictions.parquet"
imp_path = PROCESSED_DIR / "lgbm_feature_importance.parquet"
features_path = PROCESSED_DIR / "lgbm_selected_features.json"

save_booster(final_booster, model_path)

valid_predictions = pd.DataFrame(
    {
        "SK_ID_CURR": valid_df["SK_ID_CURR"].values,
        "TARGET": y_valid.values,
        "pd_proba": y_pred_valid_final,
    }
)
valid_predictions.to_parquet(preds_path, index=False)

final_importance_df.to_parquet(imp_path, index=False)

with open(features_path, "w", encoding="utf-8") as f:
    json.dump(final_features, f, ensure_ascii=False, indent=2)

print(f"final model    : used ({len(final_features)} features)")
print(f"best iteration : {final_booster.best_iteration}")
print(f"valid AUC      : {final_booster.best_score['valid']['auc']:.6f}\n")

for p in (model_path, preds_path, imp_path, features_path):
    size_kb = p.stat().st_size / 1024
    suffix = "MB" if size_kb > 1024 else "KB"
    size = size_kb / 1024 if size_kb > 1024 else size_kb
    print(f"saved : {p.name:<35} ({size:.2f} {suffix})")